# 02 — Model housing decoupling

This notebook asks whether the descriptive relationship between short-term accommodation exposure and rents survives transparent statistical controls.

The baseline model is

\[
\log R_{it}=\alpha_i+\delta_t+\beta_1 THCR_{it}+\beta_2\log Y_{it}+\beta_3 TI_{it}+\varepsilon_{it}.
\]

Municipality fixed effects absorb time-invariant local differences. Year fixed effects absorb shocks common to Portugal. Standard errors are clustered by municipality.

This remains an **associational** design unless a credible identification strategy is added.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from housing_tourism.models import (  # noqa: E402
    FixedEffectsSpec,
    build_ehpi,
    coefficient_table,
    fit_two_way_fixed_effects,
    make_baseline_counterfactual,
)

PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

## 1. Load and audit the measurement panel

In [ ]:
PANEL_PATH = PROCESSED / "municipality_housing_panel.parquet"
if not PANEL_PATH.exists():
    raise FileNotFoundError("Run Notebook 01 and complete the RNAL/municipality audit before modelling.")
panel = pd.read_parquet(PANEL_PATH)
coverage = panel.groupby("year").agg(
    municipalities=("geo_code", "nunique"),
    rent_observed=("rent_eur_m2", "count"),
    thcr_observed=("thcr", "count"),
    tourism_observed=("tourism_intensity", "count"),
)
display(coverage)

## 2. Progressive fixed-effects specifications

In [ ]:
specifications = {
    "A_thcr_only": FixedEffectsSpec(regressors=("thcr",)),
    "B_plus_income": FixedEffectsSpec(regressors=("thcr", "log_income")),
    "C_plus_tourism": FixedEffectsSpec(regressors=("thcr", "log_income", "tourism_intensity")),
}
results = {name: fit_two_way_fixed_effects(panel, spec) for name, spec in specifications.items()}
for name, result in results.items():
    print(name)
    display(coefficient_table(result).query("term in ['thcr', 'log_income', 'tourism_intensity']"))

## 3. Effect scale and lagged-exposure robustness

In [ ]:
main = results["C_plus_tourism"]
beta_thcr = float(main.params["thcr"])
for delta in (1.0, 5.0, 10.0):
    percent_change = 100.0 * (np.exp(beta_thcr * delta) - 1.0)
    print(f"+{delta:g} THCR units -> {percent_change:.2f}% conditional rent difference")

lagged = panel.sort_values(["geo_code", "year"]).copy()
lagged["thcr_lag1"] = lagged.groupby("geo_code", sort=False)["thcr"].shift(1)
lag_result = fit_two_way_fixed_effects(
    lagged, FixedEffectsSpec(regressors=("thcr_lag1", "log_income", "tourism_intensity"))
)
display(coefficient_table(lag_result))

## 4. External Housing Pressure Index

The counterfactual holds each municipality's THCR at its 2017 level while leaving the other observed covariates unchanged. This is a model-specific descriptive counterfactual, not a causal estimate of rents without Airbnb or AL.

In [ ]:
counterfactual = make_baseline_counterfactual(panel, base_year=2017)
panel_with_ehpi = panel.copy()
panel_with_ehpi["ehpi"] = build_ehpi(main, panel, counterfactual)
panel_with_ehpi.to_parquet(PROCESSED / "municipality_housing_panel_modelled.parquet", index=False)
display(panel_with_ehpi[["geo_name", "year", "thcr", "lhdi", "ehpi"]].sort_values("ehpi", ascending=False).head(20))

## 5. Interpretation discipline

Before making a substantive claim, compare defensible AL exposure definitions, lagged exposure, balanced-panel sensitivity and influential-market exclusions. If the coefficient is unstable, report the instability rather than expanding the model until significance appears.

A statement such as *Airbnb caused X% of Lisbon's rent increase* requires a separate identification design and is outside this notebook's current evidence.